# Do IMDb ratings improve the credits hybrid?

Add an IMDb audience-quality prior to the personalized model from notebook 09.
IMDb averages summarize other viewers' opinions; they are **not** individual preference labels.
The question is whether this extra signal improves retrieval of each MovieLens user's later likes.

- **Prerequisites:** run notebooks [01](01_data_exploration.ipynb),
  [04](04_plot_text_preparation.ipynb), and [09](09_credits_based_recommendations.ipynb).
  This notebook verifies and reconstructs 09's selected CF + plots + credits control.
- **Data:** official [IMDb title.ratings](https://data.imdb.com/non-commercial-datasets/)
  (`averageRating`, `numVotes`), joined through exact MovieLens IMDb IDs. No API key.
  The first run streams the ratings archive; later runs verify and reuse a small subset.
  IMDb's linked non-commercial usage terms apply.
- **Protocol:** the existing chronological 60/20/20 split, training-observed candidates,
  likes >= 4, all training-rated movies excluded, the same validation users and full positive
  denominators for Recall@10 and NDCG@10. Missing IMDb observations never remove movies.
- **Scope:** exploratory validation only. No test feedback is read and no frozen model is changed.
  Current IMDb averages and vote counts can contain opinions posted after MovieLens timestamps:
  this is a retrospective benchmark, not a historically available or leakage-free deployment estimate.
- **Outputs:** coverage, tuning grid, paired changes, per-user metrics, examples, and source hashes
  in `data/processed/per_user_chronological_60_20_20_v1/notebook10_imdb_ratings/`.

In [1]:
from pathlib import Path
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10
SEED = 42

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()), None
)
if project_root is None:
    raise FileNotFoundError("Run from inside the MovieLens project.")
raw_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to create the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} differs from the canonical split.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]
train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]
assert not train_ratings.duplicated(["userId", "movieId"]).any()
assert not validation_ratings.duplicated(["userId", "movieId"]).any()
assert set(map(tuple, train_ratings[["userId", "movieId"]].to_numpy())).isdisjoint(
    map(tuple, validation_ratings[["userId", "movieId"]].to_numpy())
)
assert (train_ratings.groupby("userId")["timestamp"].max()
        <= validation_ratings.groupby("userId")["timestamp"].min()).all()

output_dir = processed_dir / "notebook10_imdb_ratings"
output_dir.mkdir(parents=True, exist_ok=True)

def with_provenance(frame):
    saved = frame.copy()
    for column, value in {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }.items():
        saved[column] = value
    return saved

def check_provenance(frame, label):
    for column, value in {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }.items():
        assert column in frame and frame[column].eq(value).all(), (
            f"{label}: incompatible {column}; rerun its source notebook."
        )

print("Python:", platform.python_version())
print("Split:", SPLIT_ID)
print("Training / validation ratings:", len(train_ratings), "/", len(validation_ratings))
print("Outputs:", output_dir)

Python: 3.14.10
Split: per_user_chronological_60_20_20_v1
Training / validation ratings: 60255 / 20164
Outputs: C:\Users\clark\OneDrive\Documents\Projects\movie-recommendation-system\data\processed\per_user_chronological_60_20_20_v1\notebook10_imdb_ratings


## 1. Reconstruct the existing controls

Reuse 09's recorded plot weight, credit roles, and credit weight without retuning them.
Verify its split, metadata hashes, and saved per-user metrics before adding ratings.
The feature definitions below match 09: training-only vocabularies, mean cosine similarity
to liked training movies, and fallback to the base model for empty credit profiles.

In [2]:
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from scripts.prepare_imdb_ratings import prepare_ratings
from scripts.imdb_ratings_model import rating_prior, rerank_with_ratings

control_dir = processed_dir / "notebook09_credits"
control_manifest_path = control_dir / "experiment_manifest.json"
if not control_manifest_path.is_file():
    raise FileNotFoundError("Run notebook 09 first to save the credits control.")
control_manifest = json.loads(control_manifest_path.read_text(encoding="utf-8"))
for key, expected in {
    "split_id": SPLIT_ID, "split_fingerprint": split_manifest["split_fingerprint"],
    "evaluation_split": "validation", "positive_threshold": POSITIVE_THRESHOLD,
    "k": TOP_K, "test_feedback_read": False,
}.items():
    assert control_manifest[key] == expected, f"Notebook 09 has incompatible {key}."
credits_path = project_root / "data/raw/imdb/movielens_credits.json"
assert file_sha256(credits_path) == control_manifest["credits_catalog_sha256"]
credits_payload = json.loads(credits_path.read_text(encoding="utf-8"))
assert control_manifest["credits_sources"]["links_sha256"] == file_sha256(raw_dir / "links.csv")
assert control_manifest["minimum_candidate_movies_per_person"] == 2
credits = pd.DataFrame(credits_payload["movies"])
assert credits["movieId"].is_unique and set(credits["movieId"]) == set(movies["movieId"])
chosen_plot_weight = float(control_manifest["chosen_plot_weight"])
credit_variant = control_manifest["provisional_credit_variant"]
credit_roles = control_manifest["role_sets"][credit_variant]
credit_weight = float(control_manifest["provisional_credit_weight"])
print("Recorded control:", credit_variant, "plot weight:", chosen_plot_weight, "credit weight:", credit_weight)

Recorded control: cast plot weight: 0.25 credit weight: 0.25


In [3]:
user_ids = np.sort(train_ratings["userId"].unique())
movie_ids = np.sort(train_ratings["movieId"].unique())
user_index = pd.Index(user_ids)
movie_index = pd.Index(movie_ids)
n_users, n_items = len(user_ids), len(movie_ids)

X = np.zeros((n_users, n_items), dtype=np.float64)
seen = np.zeros((n_users, n_items), dtype=bool)
train_u = user_index.get_indexer(train_ratings["userId"])
train_i = movie_index.get_indexer(train_ratings["movieId"])
seen[train_u, train_i] = True
positive_rows = train_ratings["rating"].ge(POSITIVE_THRESHOLD).to_numpy()
X[train_u[positive_rows], train_i[positive_rows]] = 1.0
liked_counts = X.sum(axis=1).astype(int)
popularity_counts = np.bincount(train_i, minlength=n_items)

validation_positives = validation_ratings.loc[
    validation_ratings["rating"].ge(POSITIVE_THRESHOLD)
]
positive_ids_by_user = {
    int(user_id): set(group["movieId"].astype(int))
    for user_id, group in validation_positives.groupby("userId", sort=True)
}
evaluation_user_ids = np.array(sorted(positive_ids_by_user), dtype=int)
if len(evaluation_user_ids) == 0:
    raise ValueError("No validation users have positive feedback.")
assert user_index.get_indexer(evaluation_user_ids).min() >= 0
assert X.sum() == positive_rows.sum()
assert seen.sum() == len(train_ratings)
assert popularity_counts.sum() == len(train_ratings)

# Keep chronological liked-item order to reproduce notebook 07's averaging.
liked_columns_by_user = [
    movie_index.get_indexer(
        train_ratings.loc[
            train_ratings["userId"].eq(user_id)
            & train_ratings["rating"].ge(POSITIVE_THRESHOLD), "movieId"
        ]
    )
    for user_id in user_ids
]
print("Binary feedback shape:", X.shape)
print("Training likes:", int(X.sum()))
print("Validation-positive users:", len(evaluation_user_ids))
print("Eligible users without training likes:",
      int((liked_counts[user_index.get_indexer(evaluation_user_ids)] == 0).sum()))

Binary feedback shape: (610, 6885)
Training likes: 29862
Validation-positive users: 594
Eligible users without training likes: 1


In [4]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer, normalize

def build_role_features(records, item_ids, role, min_movies=2):
    lookup = {int(row["movieId"]): row for row in records}
    labels = [sorted(set(lookup.get(int(item_id), {}).get(role, []))) for item_id in item_ids]
    encoder = MultiLabelBinarizer(sparse_output=True)
    matrix = encoder.fit_transform(labels).astype(np.float64).tocsr()
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    keep = counts >= min_movies
    matrix = matrix[:, keep]
    names = np.asarray(encoder.classes_)[keep]
    if matrix.shape[1]:
        idf = np.log((1 + len(item_ids)) / (1 + counts[keep])) + 1
        matrix = normalize(matrix.multiply(idf).tocsr(), norm="l2")
    return matrix.tocsr(), names

def score_features(feedback, features):
    feedback = csr_matrix(feedback)
    features = csr_matrix(features)
    covered = features.getnnz(axis=1) > 0
    usable_likes = np.asarray(feedback[:, covered].sum(axis=1)).ravel()
    scores = ((feedback @ features) @ features.T).toarray()
    np.divide(scores, usable_likes[:, None], out=scores, where=usable_likes[:, None] > 0)
    assert np.isfinite(scores).all()
    return scores, usable_likes

def scale_unseen(scores, seen_mask):
    scores = np.asarray(scores, dtype=float)
    if scores.shape != seen_mask.shape or not np.isfinite(scores).all() or (scores < 0).any():
        raise ValueError("Scores must be finite, nonnegative, and match the seen mask.")
    maximum = np.where(seen_mask, 0.0, scores).max(axis=1)
    result = np.zeros_like(scores)
    np.divide(scores, maximum[:, None], out=result, where=maximum[:, None] > 0)
    return result

def blend_with_fallback(base, content, weight, seen_mask):
    if not 0 <= weight <= 1:
        raise ValueError("Blend weight must lie in [0, 1].")
    base = scale_unseen(base, seen_mask)
    content = scale_unseen(content, seen_mask)
    result = (1 - weight) * base + weight * content
    missing = np.where(seen_mask, 0.0, content).max(axis=1) <= 0
    result[missing] = base[missing]
    return result

In [5]:
def rank_unseen(scores, seen_row, counts, item_ids, k=10):
    scores = np.asarray(scores, dtype=float)
    item_ids = np.asarray(item_ids)
    if not (scores.shape == seen_row.shape == counts.shape == item_ids.shape):
        raise ValueError("Ranking arrays must have identical shapes.")
    if not np.isfinite(scores).all() or len(np.unique(item_ids)) != len(item_ids):
        raise ValueError("Ranking requires finite scores and unique item IDs.")
    candidates = np.flatnonzero(~seen_row)
    order = np.lexsort((item_ids[candidates], -counts[candidates], -scores[candidates]))
    return item_ids[candidates[order[:k]]]

def ranking_metrics(recommendations, positives, k=10):
    recommendations = list(recommendations)
    positives = set(positives)
    if not positives or len(recommendations) > k or len(set(recommendations)) != len(recommendations):
        raise ValueError("Require positives and up to k unique recommendations.")
    hits = np.array([item in positives for item in recommendations], dtype=float)
    discounts = 1 / np.log2(np.arange(2, len(recommendations) + 2))
    ideal = (1 / np.log2(np.arange(2, min(k, len(positives)) + 2))).sum()
    return int(hits.sum()), float(hits.sum() / len(positives)), float((hits * discounts).sum() / ideal)

def evaluate_scores(scores):
    rows = []
    for user_id in evaluation_user_ids:
        row = user_index.get_loc(user_id)
        recs = rank_unseen(scores[row], seen[row], popularity_counts, movie_ids, TOP_K)
        assert not set(recs) & set(movie_ids[seen[row]])
        positives = positive_ids_by_user[int(user_id)]
        hits, recall, ndcg = ranking_metrics(recs, positives, TOP_K)
        rows.append({"userId": int(user_id), "n_positives": len(positives),
                     "hits_at_10": hits, "recall_at_10": recall, "ndcg_at_10": ndcg})
    return pd.DataFrame(rows)

def mean_metrics(results):
    return {"users": len(results), "mean_recall_at_10": results["recall_at_10"].mean(),
            "mean_ndcg_at_10": results["ndcg_at_10"].mean()}

In [6]:
feature_matrices = {"cf": normalize(csr_matrix(X.T), norm="l2")}
feature_names = {}
for role in ["directors", "writers", "cast"]:
    feature_matrices[role], feature_names[role] = build_role_features(
        credits_payload["movies"], movie_ids, role
    )
candidate_movies = movies.set_index("movieId").loc[movie_ids]
genre_records = [{"movieId": int(movie_id),
                  "genres": [] if genres == "(no genres listed)" else genres.split("|")}
                 for movie_id, genres in candidate_movies["genres"].items()]
feature_matrices["genres"], feature_names["genres"] = build_role_features(
    genre_records, movie_ids, "genres", min_movies=1
)

plot_manifest_path = processed_dir / "plot_catalog_manifest.json"
if not plot_manifest_path.is_file():
    raise FileNotFoundError("Run notebook 04 first to prepare the plot catalog.")
plot_manifest = json.loads(plot_manifest_path.read_text(encoding="utf-8"))
assert plot_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
plot_path = processed_dir / "training_plot_catalog.csv"
assert file_sha256(plot_path) == plot_manifest["files"][plot_path.name]["sha256"]
plot_catalog = pd.read_csv(plot_path).set_index("movieId")
assert plot_catalog.index.is_unique and set(plot_catalog.index) == set(movie_ids)
plots = plot_catalog.loc[movie_ids, "plot"].fillna("")
plot_vectorizer = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True, norm="l2")
feature_matrices["plots"] = plot_vectorizer.fit_transform(plots).tocsr()

component_scores, profile_counts = {}, {}
feature_rows = []
for role, matrix in feature_matrices.items():
    component_scores[role], profile_counts[role] = score_features(X, matrix)
    feature_rows.append({"group": role, "features": matrix.shape[1],
                         "candidate_movies_with_features": int((matrix.getnnz(axis=1) > 0).sum()),
                         "users_with_usable_likes": int((profile_counts[role] > 0).sum())})
feature_coverage = pd.DataFrame(feature_rows)
display(feature_coverage)

,group,features,candidate_movies_with_features,users_with_usable_likes
0,cf,610,4389,606
1,directors,1379,5475,605
2,writers,3060,5688,605
3,cast,4552,6375,605
4,genres,19,6880,606
5,plots,32042,5057,606


In [7]:
assert file_sha256(plot_path) == control_manifest["plot_catalog_sha256"]
plot_scores = blend_with_fallback(component_scores["cf"], component_scores["plots"], chosen_plot_weight, seen)
credit_signal = np.mean([scale_unseen(component_scores[role], seen) for role in credit_roles], axis=0)
control_scores = blend_with_fallback(plot_scores, credit_signal, credit_weight, seen)
plot_results = evaluate_scores(plot_scores)
control_results = evaluate_scores(control_scores)
for frame, filename in [(plot_results, "control_per_user.csv"),
                        (control_results, credit_variant + "_per_user.csv")]:
    path = control_dir / filename
    assert file_sha256(path) == control_manifest["files"][filename]
    saved = pd.read_csv(path)
    check_provenance(saved, filename)
    columns = ["userId", "n_positives", "hits_at_10", "recall_at_10", "ndcg_at_10"]
    pd.testing.assert_frame_equal(frame[columns].sort_values("userId").reset_index(drop=True),
                                  saved[columns].sort_values("userId").reset_index(drop=True),
                                  check_dtype=False, atol=1e-12, rtol=0)
print("Reproduced notebook 09's per-user control results.")

Reproduced notebook 09's per-user control results.


## 2. Prepare ratings and a vote-aware quality prior

Fit **C**, the unweighted mean IMDb average among rated **training-candidate movies**, only.
Fix **m = 1,000** prior votes in advance and smooth each movie's IMDb average **R** with votes **v**:

\[
Q_i = C + \frac{v_i}{v_i + m}(R_i-C), \qquad a_i = (Q_i-C)/9.
\]

Small vote counts pull the estimate toward C; many votes leave it closer to R. This is our
regularization rule, not IMDb's internal rating or Top 250 formula. Dividing by the 1–10
rating span gives a bounded, centered adjustment. Missing/invalid ratings get Q = C and a = 0.
If no training candidate has a usable rating, stop with an explicit error.

The final score is **scaled personalized score + weight × a**. Scale each user's base by
its maximum unseen score, then try weights **0, 0.10, 0.25, 0.50, 1.0**. Higher quality can
raise a movie's ranking and below-average quality can lower it. Missing ratings leave its
scaled base score intact, although other movies' changes can still affect its rank.

In [8]:
ratings_path = prepare_ratings(project_root)
ratings_manifest = json.loads((ratings_path.parent / "ratings_manifest.json").read_text(encoding="utf-8"))
assert ratings_manifest["links_sha256"] == file_sha256(raw_dir / "links.csv")
assert ratings_manifest["catalog_sha256"] == file_sha256(ratings_path)
ratings_payload = json.loads(ratings_path.read_text(encoding="utf-8"))
ratings_catalog = pd.DataFrame(ratings_payload["movies"]).set_index("movieId")
assert ratings_catalog.index.is_unique and set(ratings_catalog.index) == set(movies["movieId"])
PRIOR_VOTES = 1000.0
prior = rating_prior(ratings_payload["movies"], movie_ids, PRIOR_VOTES)
coverage = pd.DataFrame([
    {"scope": "full_catalog", "movies": len(ratings_catalog),
     "with_ratings": int(ratings_catalog["averageRating"].notna().sum())},
    {"scope": "training_candidates", "movies": len(movie_ids),
     "with_ratings": int(prior["covered"].sum())},
])
coverage["coverage_pct"] = 100 * coverage["with_ratings"] / coverage["movies"]
display(coverage.round(2))
print("Candidate-only prior center:", round(prior["center"], 4), "Prior votes:", PRIOR_VOTES)
candidate_prior = ratings_catalog.loc[movie_ids].reset_index()
candidate_prior["smoothed_rating"] = prior["smoothed"]
candidate_prior["rating_adjustment"] = prior["adjustment"]
candidate_prior["has_imdb_rating"] = prior["covered"]

Using verified subset: title.ratings
Prepared 9,742 MovieLens rating records: C:\Users\clark\OneDrive\Documents\Projects\movie-recommendation-system\data\raw\imdb\movielens_ratings.json
Candidate-only prior center: 6.7013 Prior votes: 1000.0


,scope,movies,with_ratings,coverage_pct
0,full_catalog,9742,9719,99.76
1,training_candidates,6885,6873,99.83


## 3. Compare the IMDb-only baseline and personalized extensions

Compare both CF + plots and the recorded credits hybrid with/without ratings. Include
IMDb-only (the smoothed quality score) and the existing training-popularity baseline to
distinguish generic quality from personalization. All comparisons use identical candidates,
users, exclusions, metrics, and popularity/movie-ID tiebreaks.

Select each extension by mean validation Recall@10, preferring the smaller weight on ties.
Weight zero must reproduce all control metrics. Also report paired changes for the same users.
Reusing validation for the previous control and this tuning makes any gain exploratory.

In [9]:
RATING_WEIGHTS = [0.0, 0.10, 0.25, 0.50, 1.0]
bases = {"CF + plots": plot_scores, "CF + plots + credits": control_scores}
base_results = {"CF + plots": plot_results, "CF + plots + credits": control_results}
grid_rows, summary_rows, paired_rows = [], [], []
selected = {}
for label, scores in bases.items():
    candidates = []
    for weight in RATING_WEIGHTS:
        results = evaluate_scores(rerank_with_ratings(scores, prior["adjustment"], weight, seen))
        if weight == 0:
            pd.testing.assert_frame_equal(results, base_results[label])
        record = {"base": label, "rating_weight": weight, **mean_metrics(results)}
        grid_rows.append(record)
        candidates.append((record["mean_recall_at_10"], -weight, weight, results))
    _, _, weight, results = max(candidates, key=lambda candidate: candidate[:2])
    selected[label] = (weight, results)
    summary_rows.extend([
        {"model": label, "rating_weight": 0.0, **mean_metrics(base_results[label])},
        {"model": label + " + IMDb", "rating_weight": weight, **mean_metrics(results)},
    ])
    paired = results.merge(base_results[label], on="userId", suffixes=("_imdb", "_base"), validate="one_to_one")
    assert paired["n_positives_imdb"].equals(paired["n_positives_base"])
    delta = paired["recall_at_10_imdb"] - paired["recall_at_10_base"]
    paired_rows.append({"base": label, "rating_weight": weight, "mean_recall_change": delta.mean(),
                        "improved_users": int((delta > 0).sum()), "worsened_users": int((delta < 0).sum()),
                        "unchanged_users": int((delta == 0).sum())})
for label, signal in [("Popularity", np.zeros_like(control_scores)),
                      ("IMDb only", np.broadcast_to(prior["smoothed"], control_scores.shape))]:
    summary_rows.append({"model": label, "rating_weight": None, **mean_metrics(evaluate_scores(signal))})
rating_grid = pd.DataFrame(grid_rows)
summary = pd.DataFrame(summary_rows)
paired_summary = pd.DataFrame(paired_rows)
display(summary.round(6))
display(rating_grid.round(6))
display(paired_summary.round(6))
weight, chosen_results = selected["CF + plots + credits"]
gain = chosen_results["recall_at_10"].mean() - control_results["recall_at_10"].mean()
if weight == 0 or gain <= 0:
    print("No tested IMDb adjustment improves validation Recall@10 over the credits control.")
else:
    print(f"Exploratory IMDb weight={weight:.2f}; Recall@10 change={gain:+.6f}.")

Exploratory IMDb weight=0.10; Recall@10 change=+0.002432.


,model,rating_weight,users,mean_recall_at_10,mean_ndcg_at_10
0,CF + plots,0.0,594,0.114490,0.137954
1,CF + plots + IMDb,0.0,594,0.114490,0.137954
2,CF + plots + credits,0.0,594,0.120141,0.141095
3,CF + plots + credits + IMDb,0.1,594,0.122573,0.143483
4,Popularity,NaN,594,0.072092,0.088225
5,IMDb only,NaN,594,0.021546,0.019629


,base,rating_weight,users,mean_recall_at_10,mean_ndcg_at_10
0,CF + plots,0.00,594,0.114490,0.137954
1,CF + plots,0.10,594,0.113451,0.136767
2,CF + plots,0.25,594,0.108040,0.134366
3,CF + plots,0.50,594,0.106596,0.136400
4,CF + plots,1.00,594,0.104857,0.133635
5,CF + plots + credits,0.00,594,0.120141,0.141095
6,CF + plots + credits,0.10,594,0.122573,0.143483
7,CF + plots + credits,0.25,594,0.120875,0.143812
8,CF + plots + credits,0.50,594,0.116543,0.143186
9,CF + plots + credits,1.00,594,0.113600,0.141029


,base,rating_weight,mean_recall_change,improved_users,worsened_users,unchanged_users
0,CF + plots,0.0,0.000000,0,0,594
1,CF + plots + credits,0.1,0.002432,18,5,571


## 4. Inspect recommendations and save provenance

Show the two lowest-ID eligible validation users, chosen without inspecting gains.
Include both control rank and selected-model rank, IMDb average, votes, smoothed rating,
and the signed score adjustment. These make the quality/personalization tradeoff inspectable.
The saved manifest records sources, retrieval times, checksums, prior parameters, control
selection, complete grids, and the fact that test feedback was not read.

In [10]:
chosen_scores = rerank_with_ratings(control_scores, prior["adjustment"], weight, seen)
example_rows = []
for user_id in evaluation_user_ids[:2]:
    row = user_index.get_loc(user_id)
    baseline_order = rank_unseen(control_scores[row], seen[row], popularity_counts, movie_ids, n_items)
    base_rank = {int(item): rank for rank, item in enumerate(baseline_order, 1)}
    recs = rank_unseen(chosen_scores[row], seen[row], popularity_counts, movie_ids, TOP_K)
    for rank, item in enumerate(recs, 1):
        col = movie_index.get_loc(item)
        metadata = ratings_catalog.loc[item]
        example_rows.append({"userId": int(user_id), "rank": rank, "control_rank": base_rank[int(item)],
                             "movieId": int(item), "title": candidate_movies.loc[item, "title"],
                             "imdb_average": metadata["averageRating"], "imdb_votes": metadata["numVotes"],
                             "smoothed_rating": prior["smoothed"][col],
                             "score_adjustment": weight * prior["adjustment"][col],
                             "score": chosen_scores[row, col]})
example_recommendations = pd.DataFrame(example_rows)
display(example_recommendations.round(4))
artifacts = {
    "rating_coverage.csv": coverage, "candidate_rating_prior.csv": candidate_prior,
    "rating_weight_grid.csv": rating_grid, "validation_summary.csv": summary,
    "paired_validation_changes.csv": paired_summary, "example_recommendations.csv": example_recommendations,
    "credits_control_per_user.csv": control_results,
}
for label, (selected_weight, results) in selected.items():
    name = "credits_imdb_per_user.csv" if label.endswith("credits") else "plots_imdb_per_user.csv"
    artifacts[name] = results.assign(rating_weight=selected_weight)
for filename, frame in artifacts.items():
    saved = with_provenance(frame)
    saved["ratings_catalog_sha256"] = file_sha256(ratings_path)
    saved.to_csv(output_dir / filename, index=False)
experiment_manifest = {
    "split_id": SPLIT_ID, "split_fingerprint": split_manifest["split_fingerprint"],
    "evaluation_split": "validation", "positive_threshold": POSITIVE_THRESHOLD, "k": TOP_K,
    "ratings_sources": ratings_manifest, "control_manifest_sha256": file_sha256(control_manifest_path),
    "credits_catalog_sha256": file_sha256(credits_path), "plot_catalog_sha256": file_sha256(plot_path),
    "control_plot_weight": chosen_plot_weight, "control_credit_variant": credit_variant,
    "control_credit_weight": credit_weight, "prior_votes": PRIOR_VOTES,
    "candidate_only_prior_center": prior["center"], "rating_weights": RATING_WEIGHTS,
    "selected_weights": {label: value[0] for label, value in selected.items()},
    "score_formula": "base / maximum_unseen_base + weight * (smoothed_rating - C) / 9",
    "missing_rating_policy": "zero adjustment; retain candidate and scaled base score",
    "selection_metric": "mean validation Recall@10; smaller rating weight breaks ties",
    "temporal_limitation": "Current IMDb snapshot may include votes after MovieLens timestamps.",
    "test_feedback_read": False,
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "scipy": scipy.__version__, "scikit_learn": sklearn.__version__},
    "files": {name: file_sha256(output_dir / name) for name in sorted(artifacts)},
}
(output_dir / "experiment_manifest.json").write_text(
    json.dumps(experiment_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("Saved validation experiment:", output_dir)

Saved validation experiment: C:\Users\clark\OneDrive\Documents\Projects\movie-recommendation-system\data\processed\per_user_chronological_60_20_20_v1\notebook10_imdb_ratings


,userId,rank,control_rank,movieId,title,imdb_average,imdb_votes,smoothed_rating,score_adjustment,score
0,1,1,1,1610,"Hunt for Red October, The (1990)",7.5,235396.0,7.4966,0.0088,1.0088
1,1,2,2,1240,"Terminator, The (1984)",8.1,1023929.0,8.0986,0.0155,0.9981
2,1,3,3,2918,Ferris Bueller's Day Off (1986),7.8,422186.0,7.7974,0.0122,0.9462
3,1,4,4,541,Blade Runner (1982),8.1,896198.0,8.0984,0.0155,0.9439
4,1,5,5,1265,Groundhog Day (1993),8.0,743272.0,7.9983,0.0144,0.9419
5,1,6,8,858,"Godfather, The (1972)",9.2,2270729.0,9.1989,0.0278,0.9369
6,1,7,6,2716,Ghostbusters (a.k.a. Ghost Busters) (1984),7.8,493968.0,7.7978,0.0122,0.9330
7,1,8,7,2194,"Untouchables, The (1987)",7.8,358482.0,7.7969,0.0122,0.9315
8,1,9,9,1270,Back to the Future (1985),8.5,1486249.0,8.4988,0.0200,0.9197
9,1,10,10,589,Terminator 2: Judgment Day (1991),8.6,1309682.0,8.5986,0.0211,0.9025


## Interpretation

- Judge IMDb's contribution against the exact same credits control, including paired user changes.
  A larger IMDb weight trades preference similarity for broader audience approval; it can hurt niche tastes.
- Vote smoothing expresses confidence in the audience average, not confidence that this user will like it.
  The IMDb-only baseline is shared across users before seen exclusions.
- If zero wins, retain the control: adding a data source does not guarantee better recommendations.
- This notebook does not fit individual IMDb user factors or predict IMDb stars. It adds an aggregate
  rating prior to MovieLens personalization. Current snapshot timing and repeated validation selection
  limit claims of generalization. Confirm any apparent gain on fresh data or a new evaluation protocol.
On the recorded local snapshot, IMDb weight **0.10** raises mean validation Recall@10
from **0.120141** to **0.122573**, and NDCG@10 from **0.141095** to **0.143483**
across 594 users (18 improved, 5 worsened, 571 unchanged in Recall@10).
The CF + plots extension selects weight zero. These values describe this run;
refreshing the IMDb snapshot or the upstream control may change the results.
